In [0]:
# BLS connection + directory request
import requests
from bs4 import BeautifulSoup

source_url = "https://download.bls.gov/pub/time.series/pr/"

headers = {
    "User-Agent": "Vibhor-BLS-Project (your-email@example.com)"
}

response = requests.get(
    source_url,
    headers=headers,
    timeout=30
)

response.raise_for_status()

data = response.text

print(f"BLS directory downloaded: {len(data)} characters")

In [0]:
import re
from bs4 import BeautifulSoup

soup = BeautifulSoup(data, "html.parser")

files = []

for link in soup.find_all("a", href=True):

    href = link["href"]

    if not href.startswith("/pub/time.series/pr/"):
        continue

    file_name = link.get_text(strip=True)

    # Get the text immediately before this link
    previous_text = link.previous_sibling

    if not previous_text:
        continue

    metadata = previous_text.strip()

    # Extract date, time and file size
    match = re.search(
        r"(\d{1,2}/\d{1,2}/\d{4})\s+"
        r"(\d{1,2}:\d{2}\s+[AP]M)\s+"
        r"([\d]+)$",
        metadata
    )

    if not match:
        print(f"Metadata not found for: {file_name}")
        continue

    last_modified = f"{match.group(1)} {match.group(2)}"
    file_size = int(match.group(3))

    files.append({
        "file_name": file_name,
        "url": f"https://download.bls.gov{href}",
        "source_last_modified": last_modified,
        "source_size": file_size
    })

print(f"Files discovered: {len(files)}")

for file in files:
    print(
        file["file_name"],
        "|",
        file["source_last_modified"],
        "|",
        file["source_size"]
    )

In [0]:
# --------------------------------------------------
# Read existing files from control table
# --------------------------------------------------

existing_files = {
    row["file_name"]: {
        "source_last_modified": row["source_last_modified"],
        "source_size": row["source_size"],
        "ingestion_status": row["ingestion_status"]
    }
    for row in spark.sql("""
        SELECT
            file_name,
            source_last_modified,
            source_size,
            ingestion_status
        FROM bls.bronze.bls_file_control
        WHERE ingestion_status IN ('INGESTED', 'SOURCE_REMOVED')
    """).collect()
}


# --------------------------------------------------
# Files currently available at BLS
# --------------------------------------------------

discovered_file_names = {
    file["file_name"]
    for file in files
}


# --------------------------------------------------
# Find new or changed files
# --------------------------------------------------

files_to_ingest = []

for file in files:

    file_name = file["file_name"]

    # ----------------------------------------------
    # New file
    # ----------------------------------------------

    if file_name not in existing_files:

        files_to_ingest.append(file)

    else:

        existing = existing_files[file_name]

        # ------------------------------------------
        # File was previously removed but is
        # available again
        # ------------------------------------------

        if existing["ingestion_status"] == "SOURCE_REMOVED":

            files_to_ingest.append(file)

        # ------------------------------------------
        # File exists and source metadata changed
        # ------------------------------------------

        elif (
            existing["source_last_modified"] != file["source_last_modified"]
            or existing["source_size"] != file["source_size"]
        ):

            files_to_ingest.append(file)


# --------------------------------------------------
# Find files removed from BLS
# --------------------------------------------------

removed_files = [
    file_name
    for file_name, details in existing_files.items()
    if (
        file_name not in discovered_file_names
        and details["ingestion_status"] != "SOURCE_REMOVED"
    )
]


# --------------------------------------------------
# Display results
# --------------------------------------------------

print("Discovered files :", len(files))
print("Already tracked  :", len(existing_files))
print("Files to ingest  :", len(files_to_ingest))
print("Removed files    :", len(removed_files))


print("\nFiles to ingest:")

for file in files_to_ingest:

    print(
        file["file_name"],
        "|",
        file["source_last_modified"],
        "|",
        file["source_size"]
    )


print("\nRemoved files:")

for file_name in removed_files:

    print(file_name)

In [0]:
import requests

for file in files_to_ingest:

    file_name = file["file_name"]
    source_url = file["url"]
    source_last_modified = file["source_last_modified"]
    source_size = file["source_size"]

    tmp_path = f"/Volumes/bls/bronze/bls_raw/tmp/{file_name}"
    raw_path = f"/Volumes/bls/bronze/bls_raw/raw/{file_name}"

    print(f"\nProcessing: {file_name}")

    # --------------------------------------------------
    # 1. Download from BLS
    # --------------------------------------------------

    response = requests.get(
        source_url,
        headers=headers,
        timeout=60
    )

    response.raise_for_status()

    downloaded_size = len(response.content)

    print(f"Downloaded: {downloaded_size} bytes")


    # --------------------------------------------------
    # 2. Validate downloaded file
    # --------------------------------------------------

    if downloaded_size != source_size:

        raise Exception(
            f"Source size mismatch for {file_name}: "
            f"expected={source_size}, "
            f"downloaded={downloaded_size}"
        )

    print("Source metadata validation passed")


    # --------------------------------------------------
    # 3. Write to tmp
    # --------------------------------------------------

    with open(tmp_path, "wb") as f:
        f.write(response.content)

    print("Written to tmp")


    # --------------------------------------------------
    # 4. Validate tmp file
    # --------------------------------------------------

    actual_size = dbutils.fs.ls(tmp_path)[0].size

    if actual_size != source_size:

        raise Exception(
            f"Tmp file size mismatch for {file_name}: "
            f"expected={source_size}, "
            f"actual={actual_size}"
        )

    print("Tmp validation passed")


    # --------------------------------------------------
    # 5. Remove existing raw file
    # --------------------------------------------------

    dbutils.fs.rm(raw_path, True)


    # --------------------------------------------------
    # 6. Move tmp → raw
    # --------------------------------------------------

    dbutils.fs.mv(tmp_path, raw_path)

    print("Moved to raw")


    # --------------------------------------------------
    # 7. MERGE into control table
    # --------------------------------------------------

    spark.sql(f"""
        MERGE INTO bls.bronze.bls_file_control AS target

        USING (
            SELECT
                '{file_name}' AS file_name,
                '{source_url}' AS source_url,
                '{source_last_modified}' AS source_last_modified,
                {source_size} AS source_size,
                '{raw_path}' AS volume_path
        ) AS source

        ON target.file_name = source.file_name

        WHEN MATCHED THEN

            UPDATE SET
                target.source_url = source.source_url,
                target.source_last_modified = source.source_last_modified,
                target.source_size = source.source_size,
                target.volume_path = source.volume_path,
                target.ingestion_status = 'INGESTED',
                target.bronze_status = 'PENDING',
                target.last_ingested_at = current_timestamp()

        WHEN NOT MATCHED THEN

            INSERT (
                file_name,
                source_url,
                source_last_modified,
                source_size,
                volume_path,
                ingestion_status,
                first_ingested_at,
                last_ingested_at,
                bronze_status,
                bronze_loaded_at
            )

            VALUES (
                source.file_name,
                source.source_url,
                source.source_last_modified,
                source.source_size,
                source.volume_path,
                'INGESTED',
                current_timestamp(),
                current_timestamp(),
                'PENDING',
                NULL
            )
    """)

    print("Control table updated")


# --------------------------------------------------
# Mark files removed from BLS
# --------------------------------------------------

for file_name in removed_files:

    spark.sql(f"""
        UPDATE bls.bronze.bls_file_control
        SET
            ingestion_status = 'SOURCE_REMOVED'
        WHERE file_name = '{file_name}'
    """)

    print(f"Marked as SOURCE_REMOVED: {file_name}")


print("\nIngestion process completed.")

In [0]:
%sql
SELECT
    *
FROM bls.bronze.bls_file_control
ORDER BY file_name;